# Milestone 3 — Moteur de consolidation

Ce notebook construit le moteur qui consolide les livraisons. Pour chaque couple fournisseur × usine, il choisit un **profil de livraison** (une fréquence et des jours fixes), regroupe les besoins sur les jours de livraison, les charge dans des camions et chiffre l'économie par rapport à la baseline du milestone 2.

Plan :
1. Formalisation du problème : variables, contraintes et objectif
2. Étude des solutions possibles et choix de l'approche
3. Paramètres et données d'entrée
4. Fonction objectif et vérification des contraintes
5. Moteur d'optimisation : regroupement, sélection des profils, lissage
6. Validation des résultats
7. Économies réalisées
8. Export des sorties 1, 2 et 3

## 1. Formalisation du problème

#### Données
- $C$ : les couples fournisseur × usine ; $d_c$ : la distance du fournisseur de $c$.
- $L_c$ : les lignes de besoin du couple $c$ (silver), chacune avec une date de besoin $t_l$, un poids $w_l$ et un volume $v_l$.
- $P$ : les profils candidats. Un profil $p = (f, J)$ est une fréquence $f \in \{1, 2, 3, 5\}$ et un ensemble de jours de la semaine $J$, avec $|J| = f$.
- $\text{prix}(w, d)$ : le prix d'un envoi d'après la grille tarifaire, soit le coût de la grille × (1 + 2,3923) × 1,1943.

#### Variables de décision
- $x_{c,p} \in \{0, 1\}$ vaut 1 si le couple $c$ suit le profil $p$, avec exactement un profil par couple : $\sum_p x_{c,p} = 1$.
- La date de livraison $s_l$ de chaque ligne et le camion $k$ qui la transporte.

#### Contraintes
| Contrainte | Écriture | Spec |
|---|---|---|
| Jamais de retard | $s_l \le t_l$ | SF-06 |
| Anticipation limitée | $t_l - s_l \le 7$ jours | SF-06 |
| Livraison un jour du profil | $\text{jour}(s_l) \in J_c$ | SF-04 |
| Capacité poids | $\sum_{l \in k} w_l \le 24\,500$ kg | SF-07 |
| Capacité volume | $\sum_{l \in k} v_l \le 110{,}4$ m³ | SF-07 |
| Un couple par camion | toutes les lignes d'un camion ont le même fournisseur et la même usine | SF-03 |
| Lot non découpé | une ligne de besoin est entièrement dans un seul camion | — |
| Exhaustivité | chaque ligne est livrée exactement une fois | SF-12 |

#### Objectif
Minimiser le coût total de transport, chaque camion étant facturé comme un envoi :

$$\min \sum_{c \in C} \sum_{k \in \text{camions}(c)} \text{prix}\Big(\sum_{l \in k} w_l,\ d_c\Big)$$

Deux critères secondaires départagent les solutions : la **fréquence la plus basse** à coût équivalent (SF-05) et la **répartition des livraisons sur la semaine** dans chaque usine (lissage, SF-08).

## 2. Étude des solutions possibles

| Approche | Principe | Avantages | Limites |
|---|---|---|---|
| **Programmation linéaire en nombres entiers** (OR-Tools, MILP) | Toutes les décisions (profil, date, camion) dans un seul modèle | Optimum global prouvé | Prix en escalier selon le poids : modèle lourd (une variable par ligne × camion × date), temps de calcul élevé, résultat difficile à expliquer |
| **Métaheuristiques** (recuit simulé, génétique) | Exploration aléatoire de l'espace des solutions | Souples | Résultats non reproductibles sans graine, réglages délicats, pas plus explicable |
| **Énumération par couple + heuristique de chargement** (retenue) | Pour chaque couple, on teste les 14 profils candidats ; pour chacun, on regroupe les besoins et on charge les camions en *First Fit Decreasing* | Exact sur le choix du profil, rapide (quelques secondes), déterministe, chaque choix s'explique par un tableau de coûts | Le chargement FFD n'est pas prouvé optimal (mais il l'est presque toujours ici : la plupart des livraisons tiennent dans un seul camion) |

**Pourquoi l'énumération suffit.** Les couples sont **indépendants pour le coût** : un camion ne transporte qu'un seul couple (SF-03), et le prix d'un camion ne dépend que de son poids et de la distance du couple. Le problème se découpe donc en un sous-problème par couple. Chaque sous-problème n'a que 14 profils candidats, qu'on peut tous évaluer. Seul le **lissage** relie les couples d'une même usine ; on le traite par une règle gloutonne : on place d'abord les plus gros couples, puis chaque couple suivant sur les jours les moins chargés de son usine.

**Profils candidats** (0 = lundi, …, 4 = vendredi) :
- 1 / semaine : un jour parmi les 5 ;
- 2 / semaine : lundi-jeudi, mardi-vendredi, lundi-mercredi, mardi-jeudi, mercredi-vendredi ;
- 3 / semaine : lundi-mercredi-vendredi, lundi-mardi-jeudi, mardi-jeudi-vendredi ;
- 5 / semaine : tous les jours (c'est la situation actuelle).

**Règle d'affectation d'un besoin.** Un besoin daté du jour $t$ est livré le **dernier jour du profil qui précède ou égale $t$**. Il n'est donc jamais en retard. L'avance est au plus de 6 jours, par exemple pour un besoin du jeudi avec un profil « vendredi » : il est livré le vendredi de la semaine précédente. La limite des 7 jours est respectée par construction.

**Démarrage du planning.** Le programme est publié le 08/11/2026 : on ne peut pas livrer avant le premier jour du planning (le 09/11/2026). Les besoins de la première semaine qui tomberaient avant cette date sont donc livrés le 09/11. Ce rattrapage coûte au plus un envoi supplémentaire par couple.

## 3. Paramètres et données d'entrée

Le moteur part du **silver** (les lignes valides du pipeline du milestone 1) et des distances. Les lignes en quarantaine (poids manquant, fournisseur sans distance, ligne de plus de 24,5 t) ne sont pas consolidées : elles sont signalées à part, comme dans la baseline. Le périmètre est donc le même que celui de la baseline, et la comparaison des coûts est juste.

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

SILVER_CLEAN_PATH = "../data/silver/released_schedule_cleaned.csv"
QUARANTINE_PATH = "../data/silver/released_schedule_quarantine.csv"
DISTANCES_PATH = "../data/bronze/distances.csv"
RATE_SHEET_PATH = "../data/bronze/rate_sheet.csv"
BASELINE_PATH = "../data/baseline/baseline_camion.csv"
GOLD_DIR = "../data/gold"

POIDS_MAX = 24500         # kg
VOLUME_MAX = 110.4        # m³
ANTICIPATION_MAX = 7      # jours d'avance au maximum (SF-06)
TOLERANCE_LISSAGE = 0.05  # surcoût accepté (5 %) pour placer un couple sur des jours moins chargés

FREIGHT_VALUE_MARGIN = 2.3923  # 239,23 %
DIESEL_SURCHARGE = 1.1943

JOURS = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi"]

# Profils candidats : fréquence -> liste des jeux de jours (0 = lundi)
PROFILS = {
    1: [(0,), (1,), (2,), (3,), (4,)],
    2: [(0, 3), (1, 4), (0, 2), (1, 3), (2, 4)],
    3: [(0, 2, 4), (0, 1, 3), (1, 3, 4)],
    5: [(0, 1, 2, 3, 4)],
}

In [ ]:
silver = pd.read_csv(SILVER_CLEAN_PATH, parse_dates=["need_date"])
silver["ligne_id"] = silver.index  # même identifiant que dans la baseline (traçabilité)

distances = pd.read_csv(DISTANCES_PATH).set_index("supplier_id")["distance"]
couts = pd.read_csv(RATE_SHEET_PATH, index_col="km_max")
baseline_camion = pd.read_csv(BASELINE_PATH, parse_dates=["date"])
quarantaine = pd.read_csv(QUARANTINE_PATH)

DEBUT_HORIZON = silver["need_date"].min()  # on ne livre pas avant le premier jour du planning

silver["distance"] = silver["supplier_id"].map(distances)
assert silver["distance"].notna().all(), "un fournisseur du silver n'a pas de distance"

nb_couples = silver.groupby(["supplier_id", "plant"]).ngroups
print(f"{len(silver)} lignes de besoin, {nb_couples} couples, du {silver['need_date'].min():%d/%m/%Y} au {silver['need_date'].max():%d/%m/%Y}")
print(f"{len(quarantaine)} lignes en quarantaine, non consolidées (signalées à part)")
print(f"Baseline : {len(baseline_camion)} camions")

## 4. Fonction objectif et vérification des contraintes

#### Fonction objectif
Le prix d'un envoi se lit dans la grille tarifaire. On prend la **première tranche supérieure ou égale** à la distance et au poids (« jusqu'à »), puis on applique la marge et la surcharge diesel. Le coût d'un plan est la somme des prix de ses camions.

In [ ]:
TRANCHES_KM = [int(km) for km in couts.index if not str(km).startswith(">")]
TRANCHES_KG = [int(kg) for kg in couts.columns if not kg.startswith(">")]


def tranche(valeur, tranches):
    """Première tranche >= valeur (tarif « jusqu'à »), sinon la tranche '>max'."""
    for borne in tranches:
        if valeur <= borne:
            return str(borne)
    return f">{tranches[-1]}"


def cout_transport(poids, distance):
    """Prix d'un envoi en EUR = coût de la grille × (1 + marge) × diesel."""
    cout_base = couts.loc[tranche(distance, TRANCHES_KM), tranche(poids, TRANCHES_KG)]
    return float(round(cout_base * (1 + FREIGHT_VALUE_MARGIN) * DIESEL_SURCHARGE, 2))


def cout_plan(camions):
    """Fonction objectif : coût total d'un plan (une ligne par camion, colonnes poids et distance)."""
    return sum(cout_transport(p, d) for p, d in zip(camions["poids"], camions["distance"]))


# Exemple de la présentation : 300 km, 1 000 kg -> 335,34 EUR
assert cout_transport(1000, 300) == 335.34
print("Exemple 300 km / 1 000 kg :", cout_transport(1000, 300), "EUR")

#### Vérification des contraintes
`verifier_contraintes` prend le plan au niveau ligne (une ligne de besoin = une ligne, avec sa date de livraison et son camion). Elle renvoie un tableau avec le nombre de violations de chaque contrainte. Un plan est valide si toutes les valeurs sont à 0.

In [ ]:
def verifier_contraintes(plan, profils):
    """Compte les violations de chaque contrainte métier. Un plan valide n'a que des 0."""
    camions = plan.groupby("camion_id").agg(poids=("weight", "sum"), volume=("volume", "sum"),
                                            nb_couples=("couple", "nunique"), nb_dates=("date_livraison", "nunique"))
    avance = (plan["need_date"] - plan["date_livraison"]).dt.days
    jours_profil = plan["couple"].map(profils.set_index("couple")["jours_idx"])
    jour_ok = [j in jours or d == DEBUT_HORIZON
               for j, jours, d in zip(plan["date_livraison"].dt.day_of_week, jours_profil, plan["date_livraison"])]

    violations = {
        "SF-07 poids camion > 24,5 t": int((camions["poids"] > POIDS_MAX + 1e-9).sum()),
        "SF-07 volume camion > 110,4 m³": int((camions["volume"] > VOLUME_MAX + 1e-9).sum()),
        "SF-03 plusieurs couples dans un camion": int((camions["nb_couples"] > 1).sum()),
        "camion livré sur plusieurs dates": int((camions["nb_dates"] > 1).sum()),
        "SF-06 livraison en retard": int((avance < 0).sum()),
        "SF-06 avance > 7 jours": int((avance > ANTICIPATION_MAX).sum()),
        "SF-04 livraison hors des jours du profil": int((~pd.Series(jour_ok, index=plan.index)).sum()),
        "livraison le week-end": int((plan["date_livraison"].dt.day_of_week > 4).sum()),
    }
    return pd.Series(violations, name="violations")

## 5. Moteur d'optimisation

Le moteur se décompose en trois briques :
1. `dates_de_livraison` : applique un profil aux besoins d'un couple, en livrant chaque besoin le dernier jour du profil avant sa date ;
2. `charger_camions` : regroupe les besoins d'une même date de livraison et les charge dans des camions en *First Fit Decreasing* (lignes de la plus lourde à la plus légère, chacune dans le premier camion où elle rentre, sinon un nouveau camion) ;
3. `evaluer_profil` : enchaîne les deux et calcule le coût du couple pour ce profil.

In [ ]:
def dates_de_livraison(need_dates, jours):
    """Dernier jour du profil <= date du besoin (jamais de retard), au plus tôt DEBUT_HORIZON."""
    jour_besoin = need_dates.dt.day_of_week.to_numpy()
    # nombre de jours à remonter pour tomber sur un jour du profil (0 à 6)
    recul = np.array([min((jb - j) % 7 for j in jours) for jb in range(7)])[jour_besoin]
    livraison = need_dates - pd.to_timedelta(recul, unit="D")
    return livraison.where(livraison >= DEBUT_HORIZON, DEBUT_HORIZON)


def charger_camions(poids, volumes):
    """First Fit Decreasing : n° de camion (1, 2, ...) de chaque ligne. Les lots ne sont jamais découpés."""
    ordre = np.argsort(-np.asarray(poids), kind="stable")
    camions_ouverts, numeros = [], np.zeros(len(ordre), dtype=int)
    for i in ordre:
        p, v = poids[i], volumes[i]
        for k, (pk, vk) in enumerate(camions_ouverts):
            if pk + p <= POIDS_MAX and vk + v <= VOLUME_MAX:
                camions_ouverts[k] = (pk + p, vk + v)
                numeros[i] = k + 1
                break
        else:
            camions_ouverts.append((p, v))
            numeros[i] = len(camions_ouverts)
    return numeros


def evaluer_profil(lignes, jours, distance):
    """Applique un profil à un couple : renvoie (lignes avec date et n° de camion, camions, coût)."""
    lignes = lignes.assign(date_livraison=dates_de_livraison(lignes["need_date"], jours))
    numeros = np.zeros(len(lignes), dtype=int)
    for _, idx in lignes.groupby("date_livraison").indices.items():
        numeros[idx] = charger_camions(lignes["weight"].to_numpy()[idx], lignes["volume"].to_numpy()[idx])
    lignes["camion"] = numeros
    camions = (lignes.groupby(["date_livraison", "camion"])
                     .agg(poids=("weight", "sum"), volume=("volume", "sum"), nb_lignes=("weight", "size"))
                     .reset_index())
    camions["distance"] = distance
    camions["cout"] = [cout_transport(p, distance) for p in camions["poids"]]
    return lignes, camions, camions["cout"].sum()

#### Sélection des profils et lissage

`choisir_profils` évalue les 14 profils de chaque couple, puis choisit pour chacun :
1. les profils dont le coût est au plus 5 % au-dessus du meilleur (`TOLERANCE_LISSAGE`), **sans jamais dépasser le coût actuel** (profil 5 / semaine) : aucun couple ne doit payer plus cher qu'aujourd'hui ;
2. parmi eux, la **fréquence la plus basse** (SF-05) ;
3. parmi les jeux de jours de cette fréquence, celui qui **charge le moins les jours déjà occupés de l'usine** (lissage, SF-08). On mesure la charge par la somme des carrés du nombre moyen de camions par jour : plus elle est faible, plus la semaine est équilibrée.

Les couples sont traités du plus gros au plus petit (poids total), pour que les gros volumes soient répartis en premier. La tolérance de 5 % sert au lissage : sans elle, presque tous les couples choisiraient le lundi, car un profil « lundi » n'a jamais de rattrapage de démarrage (le 09/11/2026 est un lundi).

In [ ]:
def choisir_profils(silver, tolerance=TOLERANCE_LISSAGE):
    """Choisit un profil par couple (coût, puis fréquence, puis lissage). Renvoie (profils, plan, camions, évaluations)."""
    nb_semaines = silver["need_date"].dt.isocalendar().week.nunique()
    groupes = dict(tuple(silver.groupby(["supplier_id", "plant"])))
    ordre = sorted(groupes, key=lambda c: -groupes[c]["weight"].sum())  # plus gros couples d'abord

    charge = {}  # (usine, jour) -> nombre moyen de camions par semaine déjà planifiés
    profils, plans, camions, evaluations = [], [], [], []
    for couple in ordre:
        supplier, plant = couple
        lignes, distance = groupes[couple], distances[supplier]

        # 1. on évalue les 14 profils candidats
        candidats = []
        for frequence, jeux in PROFILS.items():
            for jours in jeux:
                l, cam, cout = evaluer_profil(lignes, jours, distance)
                par_jour = cam["date_livraison"].dt.day_of_week.value_counts() / nb_semaines
                candidats.append(dict(frequence=frequence, jours=jours, cout=cout, lignes=l, camions=cam, par_jour=par_jour))
                evaluations.append(dict(supplier_id=supplier, plant=plant, frequence=frequence,
                                        jours=" ".join(JOURS[j] for j in jours), cout=cout, nb_camions=len(cam)))

        # 2. coût acceptable, 3. fréquence la plus basse, 4. jours les moins chargés de l'usine
        cout_min = min(c["cout"] for c in candidats)
        cout_actuel = candidats[-1]["cout"]  # profil 5 / semaine = situation actuelle
        plafond = min(cout_min * (1 + tolerance), cout_actuel)  # jamais plus cher qu'aujourd'hui
        acceptables = [c for c in candidats if c["cout"] <= plafond + 1e-9]
        freq_min = min(c["frequence"] for c in acceptables)

        def charge_apres(c):
            return sum((charge.get((plant, j), 0) + c["par_jour"].get(j, 0)) ** 2 for j in range(5))

        choix = min((c for c in acceptables if c["frequence"] == freq_min), key=lambda c: (charge_apres(c), c["cout"]))
        for j, n in choix["par_jour"].items():
            charge[(plant, j)] = charge.get((plant, j), 0) + n

        couple_id = f"{supplier} × {plant}"
        profils.append(dict(couple=couple_id, supplier_id=supplier, plant=plant, distance=distance,
                            frequence=choix["frequence"], jours_idx=choix["jours"],
                            jours=", ".join(JOURS[j] for j in choix["jours"]), cout_apres=choix["cout"],
                            cout_min_possible=cout_min))
        plans.append(choix["lignes"].assign(couple=couple_id))
        camions.append(choix["camions"].assign(couple=couple_id, supplier_id=supplier, plant=plant))

    profils = pd.DataFrame(profils)
    plan = pd.concat(plans).sort_values(["supplier_id", "plant", "date_livraison", "camion"])
    camions = pd.concat(camions, ignore_index=True)
    # identifiant unique de camion, partagé entre le plan (lignes) et la table des camions
    cle = ["couple", "date_livraison", "camion"]
    camions = camions.sort_values(["plant", "date_livraison", "couple", "camion"]).reset_index(drop=True)
    camions["camion_id"] = camions.index + 1
    plan = plan.merge(camions[cle + ["camion_id"]], on=cle, how="left")
    return profils, plan, camions, pd.DataFrame(evaluations)


debut = time.perf_counter()
profils, plan, camions_apres, evaluations = choisir_profils(silver)
print(f"Moteur exécuté en {time.perf_counter() - debut:.1f} s : {len(evaluations)} profils évalués pour {len(profils)} couples")
print(f"{len(plan)} lignes livrées dans {len(camions_apres)} camions (baseline : {len(baseline_camion)})")

## 6. Validation des résultats

On vérifie le plan sous trois angles :
- **les contraintes métier** sont toutes respectées (0 violation) ;
- **l'exhaustivité** : chaque ligne du silver est livrée une fois et une seule, et les quantités, poids et volumes sont conservés pour chaque couple et chaque matériau (SF-12) ;
- **la cohérence avec la baseline** : si on impose le profil 5 / semaine à tous les couples, le moteur doit retrouver exactement le coût de la baseline du milestone 2.

In [ ]:
violations = verifier_contraintes(plan, profils)
print(violations.to_string())
assert violations.sum() == 0, "le plan viole au moins une contrainte"

In [ ]:
def valider_resultats(plan, silver, camions):
    """Exhaustivité et conservation des quantités. Lève une erreur au premier écart."""
    # chaque ligne du silver livrée une fois et une seule
    assert plan["ligne_id"].is_unique, "une ligne est livrée deux fois"
    assert set(plan["ligne_id"]) == set(silver["ligne_id"]), "une ligne n'est pas livrée"
    # conservation des quantités, poids et volumes par couple et par matériau
    cle = ["supplier_id", "plant", "material"]
    avant = silver.groupby(cle)[["schedule_quantity", "weight", "volume"]].sum()
    apres = plan.groupby(cle)[["schedule_quantity", "weight", "volume"]].sum()
    assert np.allclose(avant.sort_index(), apres.sort_index()), "quantités non conservées"
    # la table des camions est cohérente avec le détail des lignes
    assert np.isclose(camions["poids"].sum(), plan["weight"].sum())
    assert camions["camion_id"].is_unique and set(camions["camion_id"]) == set(plan["camion_id"])
    return True


valider_resultats(plan, silver, camions_apres)
print("Exhaustivité et conservation : OK")

# Cohérence avec la baseline : profil 5 / semaine pour tous les couples
cout_5j = sum(evaluer_profil(l, PROFILS[5][0], distances[s])[2] for (s, p), l in silver.groupby(["supplier_id", "plant"]))
baseline_camion["distance"] = baseline_camion["supplier_id"].map(distances)
cout_avant = cout_plan(baseline_camion)
print(f"Coût baseline (milestone 2) : {cout_avant:,.2f} EUR | moteur avec profil 5 / semaine : {cout_5j:,.2f} EUR")
assert np.isclose(cout_5j, cout_avant), "le moteur ne retrouve pas la baseline"

## 7. Économies réalisées

#### Résultat global

In [ ]:
cout_apres = camions_apres["cout"].sum()
remplissage_avant = baseline_camion["poids"] / POIDS_MAX
remplissage_apres = camions_apres["poids"] / POIDS_MAX

synthese = pd.DataFrame({
    "Baseline": [len(baseline_camion), cout_avant, remplissage_avant.median() * 100, remplissage_avant.mean() * 100],
    "Après consolidation": [len(camions_apres), cout_apres, remplissage_apres.median() * 100, remplissage_apres.mean() * 100],
}, index=["Nombre de camions", "Coût total (EUR)", "Remplissage poids médian (%)", "Remplissage poids moyen (%)"])
synthese["Écart (%)"] = (synthese["Après consolidation"] / synthese["Baseline"] - 1) * 100
print(synthese.round(1).to_string())
print(f"\nÉconomie : {cout_avant - cout_apres:,.2f} EUR sur l'horizon, soit {(1 - cout_apres / cout_avant) * 100:.1f} %")
print(f"Coût du lissage : {profils['cout_apres'].sum() - profils['cout_min_possible'].sum():,.2f} EUR "
      f"au-dessus du coût minimal de chaque couple")

In [ ]:
# Profils retenus : répartition des fréquences et des jours
print("Couples par fréquence retenue :")
print(profils["frequence"].value_counts().sort_index().rename(lambda f: f"{f} / semaine").to_string())

nb_semaines = silver["need_date"].dt.isocalendar().week.nunique()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (titre, df, col) in zip(axes, [("Baseline", baseline_camion, "date"),
                                        ("Après consolidation", camions_apres, "date_livraison")]):
    par_jour = (df.assign(jour=df[col].dt.day_of_week)
                  .pivot_table(index="jour", columns="plant", values="poids", aggfunc="size")
                  .reindex(range(5)).fillna(0) / nb_semaines)
    par_jour.index = JOURS
    par_jour.plot(kind="bar", stacked=True, ax=ax, rot=0)
    ax.set_title(f"{titre} : camions par jour de la semaine (moyenne)")
    ax.set_ylabel("Camions par jour")
axes[1].set_ylim(axes[0].get_ylim())
plt.tight_layout()
plt.show()

#### Économie par couple (sortie 3)

Pour chaque couple, on compare le coût de la baseline (une livraison par jour de besoin) au coût après consolidation.

In [ ]:
nb_semaines = silver["need_date"].dt.isocalendar().week.nunique()
avant_couple = (baseline_camion.groupby(["supplier_id", "plant"])
                               .agg(livraisons_avant=("date", "nunique"), camions_avant=("camion_id", "size"))
                               .assign(cout_avant=baseline_camion.assign(c=[cout_transport(p, d) for p, d in zip(baseline_camion["poids"], baseline_camion["distance"])])
                                                                  .groupby(["supplier_id", "plant"])["c"].sum()))
apres_couple = camions_apres.groupby(["supplier_id", "plant"]).agg(
    livraisons_apres=("date_livraison", "nunique"), camions_apres=("camion_id", "size"), poids_moyen_camion=("poids", "mean"))

sortie3 = (profils.set_index(["supplier_id", "plant"])[["distance", "frequence", "jours", "cout_apres"]]
                  .join(avant_couple).join(apres_couple).reset_index())
sortie3["livraisons_sem_avant"] = sortie3["livraisons_avant"] / nb_semaines
sortie3["camions_sem_apres"] = sortie3["camions_apres"] / nb_semaines
sortie3["remplissage_poids_moyen"] = sortie3["poids_moyen_camion"] / POIDS_MAX
sortie3["economie_eur"] = (sortie3["cout_avant"] - sortie3["cout_apres"]).round(2)  # arrondi au centime
sortie3["economie_pct"] = sortie3["economie_eur"] / sortie3["cout_avant"]
sortie3 = sortie3.sort_values("economie_eur", ascending=False).reset_index(drop=True)

assert np.isclose(sortie3["cout_avant"].sum(), cout_avant) and np.isclose(sortie3["cout_apres"].sum(), cout_apres)
print(f"{(sortie3['economie_eur'] > 0).sum()} couples gagnants, {(sortie3['economie_eur'] == 0).sum()} inchangés, "
      f"{(sortie3['economie_eur'] < 0).sum()} perdants")
sortie3.head(10)

In [ ]:
top = sortie3.head(15).iloc[::-1]
labels = top["supplier_id"] + " × " + top["plant"]
y = np.arange(len(top))
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(y + 0.2, top["cout_avant"], height=0.4, label="Baseline", color="#b0bec5")
ax.barh(y - 0.2, top["cout_apres"], height=0.4, label="Après consolidation", color="#1f77b4")
ax.set_yticks(y, labels)
ax.set_xlabel("Coût de transport sur l'horizon (EUR)")
ax.set_title("Les 15 couples qui économisent le plus")
ax.legend()
plt.tight_layout()
plt.show()

par_usine = sortie3.groupby("plant")[["cout_avant", "cout_apres", "economie_eur"]].sum()
par_usine["economie_pct"] = (par_usine["economie_eur"] / par_usine["cout_avant"] * 100).round(1)
print(par_usine.round(0).to_string())

#### Exemple de traçabilité : un camion consolidé

Chaque camion se relie aux lignes de besoin d'origine (`ligne_id` du silver). On peut donc expliquer chaque recommandation. Ci-dessous, le contenu du camion le plus lourd d'un couple passé à 1 livraison par semaine.

In [ ]:
couple_ex = sortie3.loc[sortie3["frequence"] == 1, ["supplier_id", "plant"]].iloc[0]
cam_ex = (camions_apres[(camions_apres["supplier_id"] == couple_ex["supplier_id"]) & (camions_apres["plant"] == couple_ex["plant"])]
          .sort_values("poids", ascending=False).iloc[0])
contenu = plan[plan["camion_id"] == cam_ex["camion_id"]]
print(f"Camion {cam_ex['camion_id']} : {couple_ex['supplier_id']} × {couple_ex['plant']}, livré le "
      f"{JOURS[cam_ex['date_livraison'].day_of_week].lower()} {cam_ex['date_livraison']:%d/%m/%Y}, "
      f"{cam_ex['poids']:,.0f} kg ({cam_ex['poids'] / POIDS_MAX:.1%}), {cam_ex['volume']:.1f} m³, {cam_ex['cout']:.2f} EUR")
(contenu.groupby(["material", "unloading_point"])
        .agg(besoins_regroupes=("need_date", lambda d: ", ".join(sorted(d.dt.strftime("%d/%m")))),
             quantite=("schedule_quantity", "sum"), poids=("weight", "sum"), volume=("volume", "sum"))
        .sort_values("poids", ascending=False))

## 8. Export des sorties

Les trois sorties sont écrites dans `data/gold/` :
- **Sortie 1** `sortie1_profils.csv` : le profil à appliquer à chaque couple (fréquence, jours, camions par semaine, remplissage) ;
- **Sortie 2** `sortie2_camions.csv` : le contenu de chaque camion, une ligne par ligne de besoin (matériau, quai, quantité, poids, volume, date du besoin et date de livraison), reliée au silver par `ligne_id` ;
- **Sortie 3** `sortie3_economies.csv` : le coût avant et après, et l'économie par couple en EUR et en %.

In [ ]:
os.makedirs(GOLD_DIR, exist_ok=True)

sortie1 = sortie3[["supplier_id", "plant", "distance", "livraisons_sem_avant", "frequence", "jours",
                   "camions_sem_apres", "remplissage_poids_moyen"]].sort_values(["plant", "supplier_id"])

sortie2 = plan[["camion_id", "supplier_id", "plant", "date_livraison", "ligne_id", "material", "unloading_point",
                "need_date", "schedule_quantity", "weight", "volume"]].sort_values(["camion_id", "weight"], ascending=[True, False])
sortie2 = sortie2.assign(avance_jours=(sortie2["need_date"] - sortie2["date_livraison"]).dt.days)

sortie3_export = sortie3[["supplier_id", "plant", "distance", "frequence", "jours", "camions_avant", "camions_apres",
                          "cout_avant", "cout_apres", "economie_eur", "economie_pct"]]

for table, nom in [(sortie1, "sortie1_profils.csv"), (sortie2, "sortie2_camions.csv"), (sortie3_export, "sortie3_economies.csv")]:
    path = f"{GOLD_DIR}/{nom}"
    table.round(dict.fromkeys(table.select_dtypes("float").columns, 6)).to_csv(path, index=False, date_format="%Y-%m-%d")
    print(f"Exporté : {path} ({len(table)} lignes)")

## Conclusion

Le moteur consolide les 4 251 lignes de besoin du silver en **697 camions au lieu de 1 603 (−56,5 %)**, pour un coût de transport de **86 375 EUR au lieu de 142 965 EUR, soit 56 590 EUR économisés (−39,6 %)** sur les 6 semaines du planning. Toutes les contraintes métier sont respectées (0 violation), chaque ligne est livrée une fois et une seule, et le moteur retrouve exactement le coût de la baseline quand on lui impose le profil actuel.

Ce qu'il faut retenir :
- **Tous les couples passent à 1 livraison par semaine.** Avec la grille tarifaire, le prix d'un envoi augmente beaucoup moins vite que son poids : regrouper est donc toujours rentable. Et la capacité n'est presque jamais limitante, car même regroupés sur une semaine, les besoins d'un couple tiennent le plus souvent dans un seul camion.
- **L'économie est répartie sur les 4 usines** (de −30 % pour PLT_003 à −43 % pour PLT_004). 128 couples économisent, 43 sont inchangés : ils ne livraient déjà qu'une fois par semaine au plus. Aucun couple ne paie plus cher qu'aujourd'hui.
- **Le lissage fonctionne** : les camions sont répartis presque également sur les 5 jours dans chaque usine. Il coûte 721 EUR, soit environ 1 % de l'économie. Le lundi reste un peu plus chargé, à cause du rattrapage de démarrage du 09/11.
- **Le remplissage reste faible** (9 % en moyenne) : beaucoup de couples ont de très petits volumes, et une livraison par semaine est la fréquence minimale autorisée par les profils. Le gain vient donc surtout du nombre d'envois.

Limites et suites (milestone 4) :
- les 139 lignes en quarantaine (poids manquant, SUP_00130 sans distance, ligne de plus de 24,5 t) ne sont pas consolidées ; elles sont signalées à part ;
- le tarif au-delà de 4 000 kg est appliqué par envoi, comme indiqué dans la présentation : il reste à confirmer ;
- le chargement *First Fit Decreasing* n'est pas prouvé optimal ; les tests unitaires, les cas limites et l'analyse avant/après détaillée font l'objet du milestone 4.